# JS13 Praktikum 3 - ANN Regresi Sleep Wise
Memprediksi `Sleep Duration` menggunakan neural network regression.

In [1]:
from pathlib import Path
import pickle
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPRegressor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data = pd.read_csv(Path('../Dataset/Sleep_health_and_lifestyle_dataset.csv'))
data['Sleep Disorder'] = data['Sleep Disorder'].fillna('None')
bp = data['Blood Pressure'].str.split('/', expand=True).astype(float)
data['Systolic BP'], data['Diastolic BP'] = bp[0], bp[1]
y = data['Sleep Duration']
data = data.drop(columns=['Person ID', 'Blood Pressure'])
X = data.drop(columns=['Sleep Duration', 'Sleep Disorder'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
numeric = X.select_dtypes(include=['number']).columns.tolist()
categorical = [col for col in X.columns if col not in numeric]
preprocess = ColumnTransformer([('numeric', StandardScaler(), numeric), ('categorical', OneHotEncoder(handle_unknown='ignore'), categorical)])
model = Pipeline([('preprocess', preprocess), ('ann', MLPRegressor(hidden_layer_sizes=(64, 32), activation='relu', max_iter=1500, early_stopping=True, random_state=42))])
model.fit(X_train, y_train)
pred = model.predict(X_test)
print(f'MAE: {mean_absolute_error(y_test, pred):.3f} jam')
print(f'RMSE: {mean_squared_error(y_test, pred) ** 0.5:.3f} jam')
print(f'R2: {r2_score(y_test, pred):.3f}')
with open('js13pbl_ann_regression.pkl', 'wb') as file: pickle.dump(model, file)

MAE: 0.124 jam
RMSE: 0.221 jam
R2: 0.926
